# المحاضرة 38: بناء محرك التقطيع الدلالي بلغة بايثون
## القسم الأول: الشرح النظري والمعماري الشامل

### 1. فلسفة التقطيع الدلالي ومقارنته بالتقطيع الحجمي
التقطيع المعتمد على عدد الحروف والرموز (Fixed-Size Splitting) يقسم الوثائق بآلية عمياء عن المعنى؛ فإذا انتهى الحجم المحدد في منتصف فكرة، يتم تمزيق الجملة، أو يتم دمج موضوعين متباينين في نفس القطعة لمجرد اكتمال الحجم.

الحل الهندسي هو التقطيع الدلالي (Semantic Chunking):
- تقطيع النصوص عند حدود تغير الموضوع بدلاً من عدد الحروف.
- قياس درجة التشابه بين الجمل المتتابعة باستخدام متجهات التضمين (Embeddings).
- تحديد نقاط الانفصال (Breakpoints) عندما ينخفض التشابه عن حد أدنى (Threshold).

---

### 2. الخوارزمية الرياضية خطوة بخطوة
تتكون الخوارزمية من أربع مراحل متتابعة:
1. تجزئة الوثيقة (Document Segmentation): تقسيم النص إلى قائمة جمل مستقلة $S = [S_0, S_1, \dots, S_{n-1}]$.
2. حساب المتجهات (Sentence Embeddings): توليد متجهات كثيفة لكل جملة $E = [E_0, E_1, \dots, E_{n-1}]$.
3. قياس التشابه الزوجي (Pairwise Cosine Similarity):
   $$\text{Sim}(E_i, E_{i+1}) = \frac{E_i \cdot E_{i+1}}{\|E_i\| \|E_{i+1}\|}$$
4. التجميع الحلقي وكشف نقاط الانفصال (Threshold Boundary Grouping):
   - إذا كان $\text{Sim}(E_i, E_{i+1}) \ge \text{Threshold}$، تدمج الجملة $S_{i+1}$ مع القطعة الحالية.
   - إذا انخفض التشابه عن العتبة، يتم حفظ القطعة الحالية وبدء قطعة جديدة تحتوي على $S_{i+1}$.

---

### 3. مخطط تدفق خوارزمية التقطيع الدلالي

```mermaid
flowchart TD
    subgraph Phase1 ["1. تجزئة الجمل"]
        A["مستند متعدد الموضوعات"] --> B["قائمة الجمل المفردة"]
    end

    subgraph Phase2 ["2. حساب التضمينات"]
        B --> C["SentenceTransformer('all-MiniLM-L6-v2')"]
        C --> D["مصفوفة المتجهات (384 بعداً لكل جملة)"]
    end

    subgraph Phase3 ["3. قياس التشابه الزوجي"]
        D --> E["Sim(S0, S1) = 0.8248 (ترابط عالي)"]
        D --> F["Sim(S1, S2) = 0.4487 (ترابط متوسط)"]
        D --> G["Sim(S2, S3) = 0.0225 (هبوط حاد - تغير موضوع)"]
        D --> H["Sim(S3, S4) = 0.3776 (ترابط متوسط)"]
    end

    subgraph Phase4 ["4. التجميع الحلقي بالعتبة"]
        E & F & G & H --> I{"هل Sim >= Threshold؟"}
        I -->|نعم| J["دمج الجملة في القطعة الحالية"]
        I -->|لا| K["نقطة انفصال: حفظ القطعة وبدء قطعة جديدة"]
    end
```

---

### 4. سلوك ضبط العتبة (Threshold Tuning)
- العتبة المحكمة (Threshold = 0.7): تفصل القطع عند أي تباين دقيق، وتنتج قطعاً صغيرة عالية التركيز (أنتجت 4 قطع في تجربة المحاضرة).
- العتبة المتوازنة (Threshold = 0.35): تدمج الموضوعات المترابطة معاً، وتنتج قطعتين شاملتين (قطعة لموضوعات LangChain وقطعة لموضوعات فرنسا وباريس).


## القسم الثاني: التطبيق البرمجي المتكامل
الكود البرمجي الكامل والنموذجي لتنفيذ محرك التقطيع الدلالي بلغة بايثون.

In [1]:
# 1. استدعاء المكتبات وتحميل موديل التضمين
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize lightweight local embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

embedding_dim = model.get_sentence_embedding_dimension()
print(f"[INFO] Model loaded successfully. Embedding dimension: {embedding_dim}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Model loaded successfully. Embedding dimension: 384


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_27908\2920368977.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


In [2]:
# 2. قراءة وتجزئة النص الخام إلى جمل مستقلة
sample_text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

# Split raw text by newlines and strip whitespace
sentences = [s.strip() for s in sample_text.split("\n") if s.strip()]

print(f"[INFO] Total extracted sentences: {len(sentences)}\n")
for idx, s in enumerate(sentences, 1):
    print(f"[{idx}] {s}")

[INFO] Total extracted sentences: 5

[1] LangChain is a framework for building applications with LLMs.
[2] LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
[3] You can create chains, agents, memory, and retrievers.
[4] The Eiffel Tower is located in Paris.
[5] France is a popular tourist destination.


In [3]:
# 3. حساب متجهات التضمين للجمل
embeddings = model.encode(sentences)

print(f"[INFO] Embeddings matrix shape: {embeddings.shape} -> ({len(sentences)} sentences x {embeddings.shape[1]} dimensions)")

[INFO] Embeddings matrix shape: (5, 384) -> (5 sentences x 384 dimensions)


In [4]:
# 4. قياس درجات التشابه الزوجي بين الجمل المتتابعة
print("[INFO] Pairwise Cosine Similarity between consecutive sentences:\n")

similarity_scores = []
for i in range(len(sentences) - 1):
    sim = cosine_similarity([embeddings[i]], [embeddings[i + 1]])[0][0]
    similarity_scores.append(sim)
    print(f"Similarity between sentence [{i+1}] and [{i+2}] = {sim:.4f}")
    print(f"   S_{i+1}: '{sentences[i][:45]}...'")
    print(f"   S_{i+2}: '{sentences[i+1][:45]}...'")
    if sim < 0.2:
        print("   [BREAKPOINT] Severe semantic drop detected (topic transition).\n")
    else:
        print("   [CONTINUITY] High/moderate semantic continuity.\n")

[INFO] Pairwise Cosine Similarity between consecutive sentences:

Similarity between sentence [1] and [2] = 0.8248
   S_1: 'LangChain is a framework for building applica...'
   S_2: 'LangChain provides modular abstraction to com...'
   [CONTINUITY] High/moderate semantic continuity.

Similarity between sentence [2] and [3] = 0.4487
   S_2: 'LangChain provides modular abstraction to com...'
   S_3: 'You can create chains, agents, memory, and re...'
   [CONTINUITY] High/moderate semantic continuity.

Similarity between sentence [3] and [4] = 0.0225
   S_3: 'You can create chains, agents, memory, and re...'
   S_4: 'The Eiffel Tower is located in Paris....'
   [BREAKPOINT] Severe semantic drop detected (topic transition).

Similarity between sentence [4] and [5] = 0.3776
   S_4: 'The Eiffel Tower is located in Paris....'
   S_5: 'France is a popular tourist destination....'
   [CONTINUITY] High/moderate semantic continuity.



In [5]:
# 5. خوارزمية التجميع الدلالي بالعتبة (Semantic Grouping Function)
def semantic_chunk_text(sentences, embeddings, threshold=0.7):
    """
    Group consecutive sentences into semantic chunks based on similarity threshold.
    """
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i]]

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Execute grouping with lecture threshold = 0.7
chunks_07 = semantic_chunk_text(sentences, embeddings, threshold=0.7)

print(f"[INFO] Generated chunks with threshold = 0.7 -> Total: {len(chunks_07)}\n")
for idx, chunk in enumerate(chunks_07, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{chunk}\n")

[INFO] Generated chunks with threshold = 0.7 -> Total: 4

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.

--- Chunk [2] ---
You can create chains, agents, memory, and retrievers.

--- Chunk [3] ---
The Eiffel Tower is located in Paris.

--- Chunk [4] ---
France is a popular tourist destination.



In [6]:
# 6. تجربة العتبة المتوازنة (Threshold = 0.35)
chunks_035 = semantic_chunk_text(sentences, embeddings, threshold=0.35)

print(f"[INFO] Generated chunks with threshold = 0.35 -> Total: {len(chunks_035)}\n")
for idx, chunk in enumerate(chunks_035, 1):
    print(f"--- Chunk [{idx}] ---")
    print(f"{chunk}\n")

[INFO] Generated chunks with threshold = 0.35 -> Total: 2

--- Chunk [1] ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

--- Chunk [2] ---
The Eiffel Tower is located in Paris. France is a popular tourist destination.



In [7]:
# 7. التحقق البرمجي النهائي (Assertions)
assert len(chunks_07) == 4, "Threshold 0.7 must produce exactly 4 chunks"
assert len(chunks_035) == 2, "Threshold 0.35 must group text into 2 thematic chunks"
assert "Eiffel Tower" in chunks_035[1], "Second chunk must contain Paris and Eiffel Tower"
assert "LangChain" in chunks_035[0], "First chunk must contain LangChain topics"

print("[SUCCESS] All semantic chunking assertions passed successfully.")

[SUCCESS] All semantic chunking assertions passed successfully.


---
## القسم الثالث: مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص مخصصة، أو اختبار قيم عتبة مختلفة وملاحظة سلوك التقطيع.

In [8]:
# مساحة التجربة العملية:
# جرب إدخال نصك الخاص وتطبيق دالة semantic_chunk_text بعتبة من اختيارك هنا:

